In [ ]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_s_curve

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def get_2d_dataset(dataset="moons", n_samples=8000):
    if dataset == "moons":
        x, _ = make_moons(n_samples=n_samples, noise=0.05)
        x = (x - np.array([0.5, 0.25])) * 3.5 
    elif dataset == "s_curve":
        x3, _ = make_s_curve(n_samples=n_samples, noise=0.05)
        x = x3[:, [0, 2]] * 1.5
    elif dataset == "8gaussians":
        scale = 1.0
        centers = [(1,0), (-1,0), (0,1), (0,-1), 
                   (1./np.sqrt(2), 1./np.sqrt(2)), (1./np.sqrt(2), -1./np.sqrt(2)), 
                   (-1./np.sqrt(2), 1./np.sqrt(2)), (-1./np.sqrt(2), -1./np.sqrt(2))]
        centers = np.array(centers) * scale
        indices = np.random.randint(0, 8, n_samples)
        noise = np.random.randn(n_samples, 2) * 0.08
        x = centers[indices] + noise
    else:
        raise ValueError("Unknown dataset")
    return torch.tensor(x, dtype=torch.float32)

def reward_fn(x, dataset="moons"):
    if dataset == "moons":
        return x[:, 1]**2 
    elif dataset == "s_curve":
        return torch.abs(x[:, 1]) 
    elif dataset == "8gaussians":
        return x[:, 0] * x[:, 1] 
    return torch.zeros(x.shape[0], device=x.device)

def get_ground_truth_tilted(data, dataset, beta=3.0, k=2000):
    rewards = reward_fn(data, dataset)
    weights = torch.exp(beta * rewards)
    weights = weights / weights.sum()
    indices = torch.multinomial(weights, k, replacement=True)
    return data[indices]

beta_min = 0.1
beta_max = 20.0

def beta_t(t: torch.Tensor) -> torch.Tensor:
    return beta_min + t * (beta_max - beta_min)

def int_beta(t: torch.Tensor) -> torch.Tensor:
    return beta_min * t + 0.5 * (beta_max - beta_min) * t * t

def alpha_t(t: torch.Tensor) -> torch.Tensor:
    return torch.exp(-0.5 * int_beta(t))

def sigma_t(t: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(1.0 - torch.exp(-int_beta(t)))

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, n_freqs: int = 64, embed_dim: int = 128, max_period: float = 10000.0):
        super().__init__()
        half = n_freqs
        freqs = torch.exp(-math.log(max_period) * torch.arange(0, half, dtype=torch.float32) / half)
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(2 * half, embed_dim),
            nn.SiLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        angles = t[:, None] * self.freqs[None, :] * 1000.0
        h = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)
        return self.mlp(h)

class ResBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.net(self.norm(x))

class EpsilonNet(nn.Module):
    def __init__(self, hidden: int = 256, t_embed_dim: int = 128, n_blocks: int = 4):
        super().__init__()
        self.t_enc = SinusoidalTimeEmbedding(n_freqs=64, embed_dim=t_embed_dim)
        self.proj_in = nn.Linear(2 + t_embed_dim, hidden)
        self.blocks = nn.ModuleList([ResBlock(hidden) for _ in range(n_blocks)])
        self.norm_out = nn.LayerNorm(hidden)
        self.proj_out = nn.Linear(hidden, 2)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        te = self.t_enc(t)
        h = F.silu(self.proj_in(torch.cat([x, te], dim=1)))
        for blk in self.blocks:
            h = blk(h)
        return self.proj_out(self.norm_out(h))

class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for sp, mp in zip(self.shadow.parameters(), model.parameters()):
            sp.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)

class EpsToScoreWrapper(nn.Module):
    def __init__(self, eps_net):
        super().__init__()
        self.eps_net = eps_net

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        eps = self.eps_net(x, t)
        s = sigma_t(t)[:, None]
        return -eps / s.clamp(min=1e-6)

def train_vp_sde(data_tensor, n_train_steps=10000, batch_size=2048):
    eps_model = EpsilonNet(hidden=256, t_embed_dim=128, n_blocks=4).to(device)
    opt = torch.optim.AdamW(eps_model.parameters(), lr=1e-3, weight_decay=1e-4)
    ema = EMA(eps_model, decay=0.9995)
    
    warmup_steps = 1000
    def get_lr(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / (n_train_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=get_lr)
    T_EPS = 1e-4
    eps_model.train()
    
    for it in range(1, n_train_steps + 1):
        idx = torch.randint(0, data_tensor.shape[0], (batch_size,), device=device)
        x0 = data_tensor[idx]
        t = torch.rand(batch_size, device=device) * (1.0 - T_EPS) + T_EPS
        a = alpha_t(t)[:, None]
        s = sigma_t(t)[:, None]
        eps = torch.randn_like(x0)
        xt = a * x0 + s * eps
        
        opt.zero_grad(set_to_none=True)
        loss = F.mse_loss(eps_model(xt, t), eps)
        loss.backward()
        nn.utils.clip_grad_norm_(eps_model.parameters(), 1.0)
        opt.step()
        scheduler.step()
        ema.update(eps_model)
        
        if it % 2000 == 0:
            print(f"  Iter {it:5d}/{n_train_steps} | Loss {loss.item():.5f}")

    score_model = EpsToScoreWrapper(ema.shadow).to(device)
    score_model.eval()
    return score_model

@torch.no_grad()
def reverse_sde_integrate(score_model, x, t_start, t_end, n_steps):
    T_MIN = 1e-3 
    t_end_safe = max(t_end, T_MIN)
    if t_end_safe >= t_start:
        return x
    dt = (t_end_safe - t_start) / n_steps
    t_cur = t_start
    
    for _ in range(n_steps):
        t_vec = torch.full((x.shape[0],), t_cur, device=x.device, dtype=x.dtype)
        b = beta_t(t_vec)[:, None]
        f = -0.5 * b * x
        g = torch.sqrt(b)
        s = score_model(x, t_vec)
        drift = f - (g ** 2) * s
        noise = torch.randn_like(x)
        x = x + drift * dt + g * math.sqrt(-dt) * noise
        t_cur += dt
    return x

@torch.no_grad()
def sample_uncond(score_model, n, n_steps=500):
    x = torch.randn(n, 2, device=device)
    return reverse_sde_integrate(score_model, x, t_start=1.0, t_end=1e-3, n_steps=n_steps)

datasets = ["moons", "s_curve", "8gaussians"]
results = {}

AXIS_LIMITS = {
    "moons": (-7, 7, -7, 7),
    "s_curve": (-3, 3, -5, 5),
    "8gaussians": (-2, 2, -2, 2),
}

for ds in datasets:
    print(f"\n--- {ds} ---")
    data_q1 = get_2d_dataset(ds, n_samples=8000).to(device)
    data_q1_tilted = get_ground_truth_tilted(data_q1, ds, beta=3.0, k=2000)
    
    print(f"Training VP-SDE...")
    score_model = train_vp_sde(data_q1, n_train_steps=10000, batch_size=1024)
    
    N_candidates = 8000
    print(f"Sampling {N_candidates} unconditional candidates...")
    candidates = sample_uncond(score_model, n=N_candidates, n_steps=500)
    
    results[ds] = {
        'q1': data_q1.cpu(),
        'q1_tilted': data_q1_tilted.cpu(),
        'unconditional_samples': candidates.cpu(),
        'score_model': score_model,
    }


In [ ]:
def make_gaussian_mode_reward(mu, sigma_mode=1.0):
    mu_t = torch.tensor(mu, dtype=torch.float32)
    inv_2s2 = 1.0 / (2.0 * sigma_mode ** 2)
    def reward_fn(x):
        mu_dev = mu_t.to(x.device)
        d2 = ((x - mu_dev[None, :]) ** 2).sum(dim=1)
        return torch.exp(-inv_2s2 * d2)
    return reward_fn

def make_quadrant_gate_reward(a=0.0, b=0.0, k=8.0):
    def reward_fn(x):
        gate = torch.sigmoid(k * (x[:, 0] - a)) * torch.sigmoid(k * (x[:, 1] - b))
        return gate.clamp(0.0, 1.0)
    return reward_fn

def make_box_reward(x_min=0.0, x_max=1.5, y_min=0.0, y_max=1.5, k=8.0):
    def reward_fn(x):
        gate = (torch.sigmoid(k * (x[:, 0] - x_min))
              * torch.sigmoid(k * (x_max - x[:, 0]))
              * torch.sigmoid(k * (x[:, 1] - y_min))
              * torch.sigmoid(k * (y_max - x[:, 1])))
        return gate.clamp(0.0, 1.0)
    return reward_fn

REWARD_CONFIGS = {
    "moons": {
        "fn": make_gaussian_mode_reward(mu=[0.0, 2.5], sigma_mode=1.5),
        "label": "Gaussian mode (0,2.5)",
    },
    "s_curve": {
        "fn": make_gaussian_mode_reward(mu=[0.0, 0.0], sigma_mode=1.0),
        "label": "Gaussian mode (0,0)",
    },
    "8gaussians": {
        "fn": make_box_reward(x_min=0.0, x_max=1.5, y_min=0.0, y_max=1.5, k=8.0),
        "label": "Box [0,1.5]²",
    },
}

def reward_fn_updated(x, dataset="moons"):
    return REWARD_CONFIGS[dataset]["fn"](x)

def plot_reward_heatmap(ax, ds, n_grid=200, cmap='YlOrRd', alpha=0.5, vmin=0, vmax=1):
    xl, xr, yl, yr = AXIS_LIMITS[ds]
    xx = torch.linspace(xl, xr, n_grid)
    yy = torch.linspace(yl, yr, n_grid)
    grid_x, grid_y = torch.meshgrid(xx, yy, indexing='xy')
    pts = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)
    r = reward_fn_updated(pts, dataset=ds).reshape(n_grid, n_grid).numpy()
    ax.imshow(r, extent=[xl, xr, yl, yr], origin='lower', aspect='auto',
              cmap=cmap, alpha=alpha, vmin=vmin, vmax=vmax)

k_samples = 2000
beta_param = 3.0
gt_tilted = {}

for ds in datasets:
    q1_original = results[ds]['q1'].to(device)
    rewards_gt = reward_fn_updated(q1_original, dataset=ds)
    weights_gt = torch.exp(beta_param * rewards_gt)
    weights_gt /= weights_gt.sum()
    gt_tilted[ds] = q1_original[torch.multinomial(weights_gt, k_samples, replacement=True)].cpu()

In [ ]:
from scipy.stats import norm as _norm_dist
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


class TwistPredictor:
    def __init__(self, name: str):
        self.name = name
    def fit(self, U, y):
        pass
    def predict_mean_std(self, U):
        pass


class RFPredictor(TwistPredictor):
    def __init__(self, n_estimators=500, random_state=42):
        super().__init__("RF")
        self.scaler = StandardScaler()
        self.rf = RandomForestRegressor(
            n_estimators=n_estimators, min_samples_leaf=3,
            random_state=random_state, n_jobs=-1,
        )

    def fit(self, U, y):
        Us = self.scaler.fit_transform(U)
        self.rf.fit(Us, y)

    def predict_mean_std(self, U):
        Us = self.scaler.transform(U)
        preds = np.stack(
            [tree.predict(Us) for tree in self.rf.estimators_], axis=0
        )
        m = preds.mean(axis=0)
        s = preds.std(axis=0)
        return m.astype(np.float32), np.maximum(s, 1e-6).astype(np.float32)


class TabPFNPredictor(TwistPredictor):
    def __init__(self):
        super().__init__("TabPFN")
        self.scaler = StandardScaler()
        from tabpfn import TabPFNRegressor
        _dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = TabPFNRegressor(
            device=_dev,
            ignore_pretraining_limits=True,
        )

    def fit(self, U, y):
        Us = self.scaler.fit_transform(U)
        self.model.fit(Us, y)

    def predict_mean_std(self, U):
        Us = self.scaler.transform(U)
        m = self.model.predict(Us)
        s = np.full_like(m, 0.25)
        return m.astype(np.float32), np.maximum(s, 1e-6).astype(np.float32)


ESTIMATOR_REGISTRY = {"rf": RFPredictor, "tabpfn": TabPFNPredictor}

def make_predictor(name, random_state=42):
    cls = ESTIMATOR_REGISTRY[name.lower()]
    if name.lower() == "rf":
        return cls(random_state=random_state)
    return cls()


def compute_raw_bpp_quantile(predictor, U, alpha=0.25, clip_c=2.0):
    m, s = predictor.predict_mean_std(U)
    z_alpha = _norm_dist.ppf(alpha)
    q = m + z_alpha * s
    q = q - np.mean(q)
    q = np.clip(q, -clip_c, clip_c)
    return q, m, s

def normalize_log_weights(log_w):
    lw = np.asarray(log_w, dtype=np.float64)
    lw = lw - np.max(lw)
    w = np.exp(lw)
    s = w.sum()
    if (not np.isfinite(s)) or s <= 0.0:
        return np.ones_like(w) / len(w)
    return w / s

def systematic_resample(weights):
    n = len(weights)
    cdf = np.cumsum(weights)
    positions = (np.random.rand() + np.arange(n)) / n
    idx = np.zeros(n, dtype=np.int64)
    i, j = 0, 0
    while i < n:
        if positions[i] <= cdf[j]:
            idx[i] = j
            i += 1
        else:
            j += 1
    return idx


def _weight_diagnostics_from_np(weights):
    w = np.asarray(weights, dtype=np.float64)
    w = np.clip(w, 1e-32, None)
    w = w / np.sum(w)

    ess = 1.0 / (np.sum(w ** 2) + 1e-12)
    entropy = -np.sum(w * np.log(w + 1e-12))
    perplexity = np.exp(entropy)

    return {
        "ess": float(ess),
        "ess_frac": float(ess / len(w)),
        "entropy": float(entropy),
        "perplexity_frac": float(perplexity / len(w)),
        "max_weight": float(np.max(w)),
    }


def extract_features_2d(x, t_scalar):
    x_np = x.cpu().numpy().astype(np.float32)
    t_col = np.full((x_np.shape[0], 1), t_scalar, dtype=np.float32)
    norms = np.linalg.norm(x_np, axis=1, keepdims=True)
    cross = x_np[:, 0:1] * x_np[:, 1:2]
    return np.concatenate([t_col, x_np, norms, x_np**2, cross], axis=1)

@torch.no_grad()
def reverse_sde_segment(score_model, x, t_start, t_end, n_steps):
    T_MIN = 1e-3
    t_end_safe = max(t_end, T_MIN)
    if t_end_safe >= t_start:
        return x
    dt = (t_end_safe - t_start) / n_steps
    t_cur = t_start
    for _ in range(n_steps):
        t_vec = torch.full((x.shape[0],), t_cur, device=x.device, dtype=x.dtype)
        b = beta_t(t_vec)[:, None]
        g = torch.sqrt(b)
        s = score_model(x, t_vec)
        drift = -0.5 * b * x - (g ** 2) * s
        x = x + drift * dt + g * math.sqrt(-dt) * torch.randn_like(x)
        t_cur += dt
    return x


def sample_bpp_2d(
    score_model,
    reward_func,
    dataset_name,
    estimator_name="rf",
    n_particles=2000,
    n_warm=2000,
    resample_times=(0.70, 0.50, 0.30, 0.15, 0.05),
    lmbda=7.0,
    alpha=0.25,
    clip_c=2.0,
    tau_max=1.0,
    total_sde_steps=500,
    return_diagnostics=False,
    return_weight_history=False,
):
    rt_sorted = sorted([t for t in resample_times if 0 < t < 1], reverse=True)
    T_MIN = 1e-3
    boundaries = [1.0] + rt_sorted + [T_MIN]
    total_range = 1.0 - T_MIN
    seg_steps = [max(1, int(round(total_sde_steps * (a - b) / total_range)))
                 for a, b in zip(boundaries[:-1], boundaries[1:])]

    print(f"  [{estimator_name.upper()} alpha={alpha}] Phase 1: {n_warm} warm-start trajectories...")
    all_features, all_labels = [], []
    remaining = n_warm
    while remaining > 0:
        bs = min(512, remaining)
        x = torch.randn(bs, 2, device=device)
        feat_blocks = []
        for seg_i in range(len(seg_steps)):
            x = reverse_sde_segment(score_model, x,
                                    t_start=boundaries[seg_i],
                                    t_end=boundaries[seg_i + 1],
                                    n_steps=seg_steps[seg_i])
            if seg_i < len(seg_steps) - 1:
                feat_blocks.append(extract_features_2d(x, boundaries[seg_i + 1]))
        rewards = reward_func(x, dataset=dataset_name).cpu().numpy()
        y = lmbda * rewards
        for fb in feat_blocks:
            all_features.append(fb)
            all_labels.append(y[:len(fb)])
        remaining -= bs

    U_train = np.concatenate(all_features, axis=0)
    y_train = np.concatenate(all_labels, axis=0)

    print(f"  [{estimator_name.upper()} alpha={alpha}] Phase 2: fitting on {len(y_train)} samples...")
    predictor = make_predictor(estimator_name, random_state=42)
    predictor.fit(U_train, y_train)
    mu_chk, _ = predictor.predict_mean_std(U_train[:min(500, len(U_train))])
    y_chk = y_train[:min(500, len(y_train))]
    r2 = float(1.0 - np.var(mu_chk - y_chk) / (np.var(y_chk) + 1e-12))
    print(f"  [{estimator_name.upper()} alpha={alpha}] R^2 = {r2:.3f}")

    print(f"  [{estimator_name.upper()} alpha={alpha}] Phase 3: guided {n_particles} particles...")
    x = torch.randn(n_particles, 2, device=device)
    ess_hist = []
    diag_hist = []
    weight_hist = []
    for seg_i in range(len(seg_steps)):
        x = reverse_sde_segment(score_model, x,
                                t_start=boundaries[seg_i],
                                t_end=boundaries[seg_i + 1],
                                n_steps=seg_steps[seg_i])
        if seg_i < len(seg_steps) - 1:
            U = extract_features_2d(x, boundaries[seg_i + 1])
            q_raw, _, _ = compute_raw_bpp_quantile(predictor, U, alpha=alpha, clip_c=clip_c)
            log_potential = tau_max * q_raw
            w = normalize_log_weights(log_potential)

            weight_diag = _weight_diagnostics_from_np(w)
            ess_hist.append(weight_diag["ess"])

            if return_weight_history:
                weight_hist.append({
                    "segment_idx": int(seg_i),
                    "t": float(boundaries[seg_i + 1]),
                    "weights": w.astype(np.float32).copy(),
                })

            idx = systematic_resample(w)
            unique_parent_frac = float(np.unique(idx).size / len(idx))
            diag_hist.append({
                "segment_idx": int(seg_i),
                "t": float(boundaries[seg_i + 1]),
                "did_resample": True,
                "unique_parent_frac": unique_parent_frac,
                **weight_diag,
            })

            x = x[torch.from_numpy(idx).long().to(device)]

    print(f"  [{estimator_name.upper()} alpha={alpha}] Mean ESS = {np.mean(ess_hist):.1f}/{n_particles}")
    if return_diagnostics and return_weight_history:
        return x.cpu(), diag_hist, weight_hist
    if return_diagnostics:
        return x.cpu(), diag_hist
    if return_weight_history:
        return x.cpu(), weight_hist
    return x.cpu()


alphas = [0.25, 0.5]
estimators = ["rf", "tabpfn"]
bpp_results = {}

for ds in datasets:
    print(f"\n{'='*40} {ds} {'='*40}")
    sm = results[ds]['score_model']
    for est_name in estimators:
        for alpha_val in alphas:
            key = (ds, est_name, alpha_val)
            bpp_results[key] = sample_bpp_2d(
                score_model=sm,
                reward_func=reward_fn_updated,
                dataset_name=ds,
                estimator_name=est_name,
                n_particles=2000,
                n_warm=500,
                resample_times=(0.70, 0.50, 0.30, 0.15, 0.05),
                lmbda=7.0,
                alpha=alpha_val,
                clip_c=5.0,
                tau_max=1.0,
                total_sde_steps=200,
            )


In [ ]:
from enum import Enum


class PotentialType(Enum):
    DIFF = "diff"
    MAX = "max"
    ADD = "add"
    RT = "rt"


def _weight_diagnostics_from_torch(w):
    w = w.float()
    w = torch.clamp(w, min=1e-32)
    w = w / w.sum()

    ess = 1.0 / (w.pow(2).sum() + 1e-12)
    entropy = -(w * torch.log(w + 1e-12)).sum()
    perplexity = torch.exp(entropy)

    return {
        "ess": float(ess.item()),
        "ess_frac": float((ess / len(w)).item()),
        "entropy": float(entropy.item()),
        "perplexity_frac": float((perplexity / len(w)).item()),
        "max_weight": float(w.max().item()),
    }


class FKD2D:
    def __init__(
        self,
        *,
        potential_type,
        lmbda,
        num_particles,
        adaptive_resampling,
        resample_frequency,
        resampling_t_start,
        resampling_t_end,
        time_steps,
        reward_min_value=0.0,
        adaptive_resample_at_end=False,
        device=torch.device("cuda"),
        track_weight_history=False,
    ):
        self.potential_type = PotentialType(potential_type)
        self.lmbda = lmbda
        self.num_particles = num_particles
        self.adaptive_resampling = adaptive_resampling
        self.adaptive_resample_at_end = adaptive_resample_at_end
        self.resample_frequency = resample_frequency
        self.resampling_t_start = resampling_t_start
        self.resampling_t_end = resampling_t_end
        self.time_steps = time_steps
        self.device = device

        self.population_rs = (
            torch.ones(self.num_particles, device=self.device) * reward_min_value
        )
        self.log_product_of_potentials = torch.zeros(
            self.num_particles, device=self.device
        )
        self._last_idx_sampled = -1
        self._reached_terminal_sample = False
        self.resample_stats = []
        self.track_weight_history = track_weight_history
        self.weight_history = []

        terminal_idx = self.time_steps - 1
        interval = np.arange(
            self.resampling_t_start, self.resampling_t_end + 1,
            self.resample_frequency,
        )
        interval = interval[np.abs(interval - terminal_idx) > 1]
        interval = np.append(interval, terminal_idx)
        self.resampling_interval = interval

    def resample(self, *, sampling_idx, rewards):
        if sampling_idx <= self._last_idx_sampled:
            raise ValueError(
                f"Sampling index {sampling_idx} must be greater than last "
                f"sampled index {self._last_idx_sampled}."
            )
        self._last_idx_sampled = sampling_idx

        at_terminal_sample = sampling_idx == self.time_steps - 1
        self._reached_terminal_sample = at_terminal_sample

        if sampling_idx not in self.resampling_interval:
            return None

        rs_candidates = torch.tensor(
            rewards, dtype=torch.float32, device=self.device
        )

        rs_raw = rs_candidates.clone()

        if self.potential_type == PotentialType.MAX:
            rs_candidates = torch.max(rs_candidates, self.population_rs)
            log_w = self.lmbda * rs_candidates
        elif self.potential_type == PotentialType.ADD:
            rs_candidates = rs_candidates + self.population_rs
            log_w = self.lmbda * rs_candidates
        elif self.potential_type == PotentialType.DIFF:
            diffs = rs_candidates - self.population_rs
            log_w = self.lmbda * diffs
        elif self.potential_type == PotentialType.RT:
            log_w = self.lmbda * rs_candidates
        else:
            raise ValueError(
                f"potential_type {self.potential_type} not recognized"
            )

        if at_terminal_sample:
            if self.potential_type in (
                PotentialType.MAX, PotentialType.ADD, PotentialType.RT,
            ):
                log_w = (
                    self.lmbda * rs_raw
                    - self.log_product_of_potentials
                )

        log_w_raw = log_w.clone()

        log_w_shifted = log_w - log_w.max()
        w = torch.exp(log_w_shifted)

        w = torch.clamp(w, 0, 1e10)
        w[torch.isnan(w)] = 0.0
        if w.sum() == 0:
            w = torch.ones_like(w)

        weight_diag = _weight_diagnostics_from_torch(w)
        w_norm = (w / w.sum()).detach().cpu().numpy().astype(np.float32)

        if self.adaptive_resampling or (
            at_terminal_sample and self.adaptive_resample_at_end
        ):
            if weight_diag["ess"] < 0.5 * self.num_particles:
                indices = torch.multinomial(
                    w, num_samples=self.num_particles, replacement=True
                )
                self.population_rs = rs_candidates[indices]
                self.log_product_of_potentials = (
                    self.log_product_of_potentials[indices]
                    + log_w_raw[indices]
                )
                unique_parent_frac = float(
                    torch.unique(indices).numel() / self.num_particles
                )
                self.resample_stats.append({
                    "sampling_idx": int(sampling_idx),
                    "at_terminal": bool(at_terminal_sample),
                    "did_resample": True,
                    "unique_parent_frac": unique_parent_frac,
                    **weight_diag,
                })
                if self.track_weight_history:
                    self.weight_history.append({
                        "sampling_idx": int(sampling_idx),
                        "at_terminal": bool(at_terminal_sample),
                        "did_resample": True,
                        "weights": w_norm.copy(),
                    })
                return indices.cpu().numpy()
            else:
                self.population_rs = rs_candidates
                self.resample_stats.append({
                    "sampling_idx": int(sampling_idx),
                    "at_terminal": bool(at_terminal_sample),
                    "did_resample": False,
                    "unique_parent_frac": 1.0,
                    **weight_diag,
                })
                if self.track_weight_history:
                    self.weight_history.append({
                        "sampling_idx": int(sampling_idx),
                        "at_terminal": bool(at_terminal_sample),
                        "did_resample": False,
                        "weights": w_norm.copy(),
                    })
                return None
        else:
            indices = torch.multinomial(
                w, num_samples=self.num_particles, replacement=True
            )
            self.population_rs = rs_candidates[indices]
            self.log_product_of_potentials = (
                self.log_product_of_potentials[indices]
                + log_w_raw[indices]
            )
            unique_parent_frac = float(torch.unique(indices).numel() / self.num_particles)
            self.resample_stats.append({
                "sampling_idx": int(sampling_idx),
                "at_terminal": bool(at_terminal_sample),
                "did_resample": True,
                "unique_parent_frac": unique_parent_frac,
                **weight_diag,
            })
            if self.track_weight_history:
                self.weight_history.append({
                    "sampling_idx": int(sampling_idx),
                    "at_terminal": bool(at_terminal_sample),
                    "did_resample": True,
                    "weights": w_norm.copy(),
                })
            return indices.cpu().numpy()


@torch.no_grad()
def predict_x0_tweedie(score_model, x_t, t_scalar):
    t_vec = torch.full(
        (x_t.shape[0],), t_scalar, device=x_t.device, dtype=x_t.dtype
    )
    s = score_model(x_t, t_vec)
    a = alpha_t(t_vec)[:, None]
    sig2 = sigma_t(t_vec)[:, None] ** 2
    return (x_t + sig2 * s) / a.clamp(min=1e-6)


@torch.no_grad()
def reverse_sde_step(score_model, x, t_cur, dt):
    t_vec = torch.full((x.shape[0],), t_cur, device=x.device, dtype=x.dtype)
    b = beta_t(t_vec)[:, None]
    g = torch.sqrt(b)
    s = score_model(x, t_vec)
    drift = -0.5 * b * x - (g ** 2) * s
    noise = torch.randn_like(x)
    return x + drift * dt + g * math.sqrt(-dt) * noise


@torch.no_grad()
def sample_fk_2d(
    score_model,
    reward_func,
    dataset_name,
    potential_type="diff",
    n_particles=2000,
    lmbda=10.0,
    resample_frequency=20,
    adaptive_resampling=False,
    adaptive_resample_at_end=False,
    reward_min_value=0.0,
    num_steps=200,
    return_diagnostics=False,
    return_weight_history=False,
):
    T_MIN = 1e-3
    timesteps = torch.linspace(1.0, T_MIN, num_steps + 1, device=device)
    dt = (T_MIN - 1.0) / num_steps

    fkd = FKD2D(
        potential_type=potential_type,
        lmbda=lmbda,
        num_particles=n_particles,
        adaptive_resampling=adaptive_resampling,
        adaptive_resample_at_end=adaptive_resample_at_end,
        resample_frequency=resample_frequency,
        resampling_t_start=-1,
        resampling_t_end=num_steps + 1,
        time_steps=num_steps + 1,
        reward_min_value=reward_min_value,
        device=device,
        track_weight_history=return_weight_history,
    )

    x = torch.randn(n_particles, 2, device=device)
    resample_count = 0

    for i in range(num_steps + 1):
        t = timesteps[i].item()
        x = reverse_sde_step(score_model, x, t, dt)
        t_after = max(t + dt, T_MIN)
        x0_hat = predict_x0_tweedie(score_model, x, t_after)
        rewards = reward_func(x0_hat, dataset=dataset_name).cpu().numpy()

        resample_indices = fkd.resample(
            sampling_idx=i,
            rewards=rewards,
        )

        if resample_indices is not None:
            x = x[torch.from_numpy(resample_indices).long().to(device)]
            resample_count += 1

    assert fkd._reached_terminal_sample, "Terminal step was not reached!"
    print(f"  [FK-{potential_type.upper()}] Done. Resampled {resample_count} times.")
    if return_diagnostics and return_weight_history:
        return x.cpu(), fkd.resample_stats, fkd.weight_history
    if return_diagnostics:
        return x.cpu(), fkd.resample_stats
    if return_weight_history:
        return x.cpu(), fkd.weight_history
    return x.cpu()


potential_types = ["diff", "max", "add", "rt"]
fk_results = {}

for ds in datasets:
    print(f"\n{'='*40} {ds} {'='*40}")
    sm = results[ds]['score_model']
    for pt in potential_types:
        key = (ds, pt)
        fk_results[key] = sample_fk_2d(
            score_model=sm,
            reward_func=reward_fn_updated,
            dataset_name=ds,
            potential_type=pt,
            n_particles=2000,
            lmbda=10.0,
            resample_frequency=20,
            adaptive_resampling=False,
            adaptive_resample_at_end=False,
            reward_min_value=0.0,
            num_steps=200,
        )


In [ ]:
row_specs = [
    ("gt",              '#8b0000', "Ground Truth"),
    ("uncond",          '#005f99', "Unconditional"),
    ("bpp_rf_0.25",     '#2e7d32', "BPP-RF (alpha=0.25)"),
    ("bpp_rf_0.5",      '#66bb6a', "BPP-RF (alpha=0.5)"),
    ("bpp_tabpfn_0.25", '#e65100', "BPP-TabPFN (alpha=0.25)"),
    ("bpp_tabpfn_0.5",  "#dd8e18", "BPP-TabPFN (alpha=0.5)"),
    ("fk_diff",         '#1565c0', "FK-DIFF"),
    ("fk_max",          '#5e35b1', "FK-MAX"),
    ("fk_add",          '#00838f', "FK-ADD"),
    ("fk_rt",           '#c62828', "FK-RT"),
]

def _get_data(tag, ds):
    if tag == "gt":
        return gt_tilted[ds]
    elif tag == "uncond":
        return results[ds]['unconditional_samples'][:2000]
    elif tag.startswith("bpp_"):
        _, est, alpha_str = tag.split("_", 2)
        return bpp_results[(ds, est, float(alpha_str))]
    elif tag.startswith("fk_"):
        pt = tag.split("_", 1)[1]
        return fk_results[(ds, pt)]

n_rows = len(row_specs)
n_cols = len(datasets)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.0 * n_rows))
plt.subplots_adjust(hspace=0.35, wspace=0.3)

for row, (tag, color, label) in enumerate(row_specs):
    for col, ds in enumerate(datasets):
        ax = axes[row, col]
        xl, xr, yl, yr = AXIS_LIMITS[ds]

        plot_reward_heatmap(ax, ds, cmap="Purples", alpha=0.35, vmin=0, vmax=1)

        data = _get_data(tag, ds)
        ax.scatter(data[:, 0], data[:, 1], s=4, alpha=0.7, color=color)
        ax.set_xlim(xl, xr)
        ax.set_ylim(yl, yr)
        ax.set_title(f"{ds} — {label}", fontsize=10)
        ax.grid(True, alpha=0.2)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()


In [ ]:
import pandas as pd

TAU = 0.80


def _as_cpu_float_tensor(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().float()
    return torch.tensor(x, dtype=torch.float32)


def _rbf_mmd2(x, y, sigma=None, max_points=2000):
    x = _as_cpu_float_tensor(x)
    y = _as_cpu_float_tensor(y)

    n = min(x.shape[0], y.shape[0], max_points)
    g = torch.Generator().manual_seed(0)
    x = x[torch.randperm(x.shape[0], generator=g)[:n]]
    y = y[torch.randperm(y.shape[0], generator=g)[:n]]

    z = torch.cat([x, y], dim=0)
    if sigma is None:
        d = torch.cdist(z, z, p=2)
        d = d[d > 0]
        sigma = torch.median(d) if d.numel() > 0 else torch.tensor(1.0)
    sigma = torch.clamp(sigma, min=1e-6)

    gamma = 1.0 / (2.0 * sigma ** 2)
    k_xx = torch.exp(-gamma * torch.cdist(x, x, p=2).pow(2))
    k_yy = torch.exp(-gamma * torch.cdist(y, y, p=2).pow(2))
    k_xy = torch.exp(-gamma * torch.cdist(x, y, p=2).pow(2))

    return float(k_xx.mean() + k_yy.mean() - 2.0 * k_xy.mean())


def _method_samples(ds):
    return {
        "gt": gt_tilted[ds],
        "uncond": results[ds]["unconditional_samples"][:2000],
        "bpp_rf_0.25": bpp_results[(ds, "rf", 0.25)],
        "bpp_rf_0.5": bpp_results[(ds, "rf", 0.5)],
        "bpp_tabpfn_0.25": bpp_results[(ds, "tabpfn", 0.25)],
        "bpp_tabpfn_0.5": bpp_results[(ds, "tabpfn", 0.5)],
        "fk_diff": fk_results[(ds, "diff")],
        "fk_max": fk_results[(ds, "max")],
        "fk_add": fk_results[(ds, "add")],
        "fk_rt": fk_results[(ds, "rt")],
    }


label_map = {k: lbl for k, _, lbl in row_specs} if "row_specs" in globals() else {}

records = []
for ds in datasets:
    gt = _as_cpu_float_tensor(gt_tilted[ds])
    for tag, samples in _method_samples(ds).items():
        x = _as_cpu_float_tensor(samples)
        r = reward_fn_updated(x, dataset=ds)
        records.append({
            "dataset": ds,
            "method": label_map.get(tag, tag),
            "mean_reward": float(r.mean().item()),
            "SR@tau_%": float((r >= TAU).float().mean().item() * 100.0),
            "MMD2_RBF_to_gt_tilted": _rbf_mmd2(x, gt),
        })

metrics_df = pd.DataFrame(records)
metrics_df = metrics_df.sort_values(["dataset", "MMD2_RBF_to_gt_tilted"]).reset_index(drop=True)

print(f"Success Rate (SR@tau) uses tau = {TAU:.2f}")
display(metrics_df)


In [ ]:
import pandas as pd
from matplotlib.gridspec import GridSpec


def set_all_seeds(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


BPP_METHODS_VIZ = [
    ("rf", 0.25, "BPP-RF (alpha=0.25)"),
    ("rf", 0.50, "BPP-RF (alpha=0.50)"),
    ("tabpfn", 0.25, "BPP-TabPFN (alpha=0.25)"),
    ("tabpfn", 0.50, "BPP-TabPFN (alpha=0.50)"),
]
FK_METHODS_VIZ = [
    ("diff", "FK-DIFF"),
    ("max", "FK-MAX"),
    ("add", "FK-ADD"),
    ("rt", "FK-RT"),
]

# Runtime controls
WEIGHT_VIZ_DATASETS = datasets
WEIGHT_VIZ_PARTICLES = 2000
WEIGHT_VIZ_STEPS = 120
WEIGHT_VIZ_BPP_WARM = 400
WEIGHT_VIZ_RESAMPLE_TIMES = (0.70, 0.50, 0.30, 0.15, 0.05)


def _sorted_weight_matrix(weight_history):
    if len(weight_history) == 0:
        return None
    rows = []
    for ev in weight_history:
        w = np.asarray(ev["weights"], dtype=np.float64)
        w_min, w_max = w.min(), w.max()
        if w_max - w_min > 1e-12:
            w = (w - w_min) / (w_max - w_min)
        else:
            w = np.ones_like(w)  # uniform if all weights identical
        rows.append(np.sort(w)[::-1])
    return np.stack(rows, axis=0)


def _topk_mass(W, k):
    k_eff = min(k, W.shape[1])
    return W[:, :k_eff].sum(axis=1)


def _collapse_row(dataset, method_label, weight_history):
    W = _sorted_weight_matrix(weight_history)
    if W is None:
        return {
            "dataset": dataset,
            "method": method_label,
            "n_weighting_events": 0,
            "peak_top1_mass": np.nan,
            "peak_top10_mass": np.nan,
            "final_top1_mass": np.nan,
            "final_top10_mass": np.nan,
        }
    top1 = _topk_mass(W, 1)
    top10 = _topk_mass(W, 10)
    return {
        "dataset": dataset,
        "method": method_label,
        "n_weighting_events": int(W.shape[0]),
        "peak_top1_mass": float(top1.max()),
        "peak_top10_mass": float(top10.max()),
        "final_top1_mass": float(top1[-1]),
        "final_top10_mass": float(top10[-1]),
    }


def _plot_weight_rank_heatmaps(dataset, bpp_panels, fk_panels):
    n_data_cols = 4
    fig = plt.figure(figsize=(4.0 * n_data_cols + 0.6, 4.2 * 2))
    gs = GridSpec(
        2, n_data_cols + 1,
        figure=fig,
        width_ratios=[1] * n_data_cols + [0.04],
        hspace=0.45,
        wspace=0.28,
    )

    axes = np.array([
        [fig.add_subplot(gs[r, c]) for c in range(n_data_cols)]
        for r in range(2)
    ])
    cax = fig.add_subplot(gs[:, n_data_cols])

    row_configs = [
        (bpp_panels, "BPP"),
        (fk_panels, "FK"),
    ]

    im = None
    for row_idx, (panels, row_label) in enumerate(row_configs):
        for col_idx, (label, weight_history) in enumerate(panels):
            ax = axes[row_idx, col_idx]
            W = _sorted_weight_matrix(weight_history)
            if W is None:
                ax.text(0.5, 0.5, "No weighting events", ha="center", va="center")
                ax.set_title(label, fontsize=9)
                ax.set_axis_off()
                continue

            im = ax.imshow(
                W, aspect="auto", origin="upper",
                cmap="Blues", vmin=0.0, vmax=1.0,
            )

            top1_peak = float(_topk_mass(W, 1).max())
            top10_peak = float(_topk_mass(W, 10).max())
            ax.set_title(f"{label}")
            ax.set_xlabel(f"Particle rank (high -> low)")
            ax.set_xticks([0, W.shape[1] // 2, W.shape[1] - 1])
            ax.set_xticklabels(["0", f"{W.shape[1] // 2}", f"{W.shape[1] - 1}"])

        axes[row_idx, 0].set_ylabel(f"{row_label}\nWeighting event index")

    if im is not None:
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Normalized particle weight", labelpad=6)

    plt.suptitle(f"Particle Weights - {dataset}")
    plt.savefig(f"weight_collapse_{dataset}.png", bbox_inches="tight", dpi=300)


weight_histories = {}
collapse_rows = []

for ds in WEIGHT_VIZ_DATASETS:
    print(f"\nCollecting weight trajectories for dataset: {ds}")
    sm = results[ds]["score_model"]

    for method_idx, (est_name, alpha_val, label) in enumerate(BPP_METHODS_VIZ):
        set_all_seeds(4000 + 97 * method_idx + 1000 * WEIGHT_VIZ_DATASETS.index(ds))
        _, w_hist = sample_bpp_2d(
            score_model=sm,
            reward_func=reward_fn_updated,
            dataset_name=ds,
            estimator_name=est_name,
            n_particles=WEIGHT_VIZ_PARTICLES,
            n_warm=WEIGHT_VIZ_BPP_WARM,
            resample_times=WEIGHT_VIZ_RESAMPLE_TIMES,
            lmbda=7.0,
            alpha=alpha_val,
            clip_c=5.0,
            tau_max=1.0,
            total_sde_steps=WEIGHT_VIZ_STEPS,
            return_weight_history=True,
        )
        weight_histories[(ds, label)] = w_hist
        collapse_rows.append(_collapse_row(ds, label, w_hist))

    for method_idx, (pt, label) in enumerate(FK_METHODS_VIZ):
        set_all_seeds(7000 + 97 * method_idx + 1000 * WEIGHT_VIZ_DATASETS.index(ds))
        _, w_hist = sample_fk_2d(
            score_model=sm,
            reward_func=reward_fn_updated,
            dataset_name=ds,
            potential_type=pt,
            n_particles=WEIGHT_VIZ_PARTICLES,
            lmbda=10.0,
            resample_frequency=20,
            adaptive_resampling=False,
            adaptive_resample_at_end=False,
            reward_min_value=0.0,
            num_steps=WEIGHT_VIZ_STEPS,
            return_weight_history=True,
        )
        weight_histories[(ds, label)] = w_hist
        collapse_rows.append(_collapse_row(ds, label, w_hist))


weight_collapse_df = pd.DataFrame(collapse_rows)
weight_collapse_df = weight_collapse_df.sort_values(
    ["dataset", "peak_top1_mass", "peak_top10_mass"],
    ascending=[True, False, False],
).reset_index(drop=True)

print("Weight-collapse summary:")
display(weight_collapse_df)

for ds in WEIGHT_VIZ_DATASETS:
    bpp_panels = [(lbl, weight_histories[(ds, lbl)]) for _, _, lbl in BPP_METHODS_VIZ]
    fk_panels = [(lbl, weight_histories[(ds, lbl)]) for _, lbl in FK_METHODS_VIZ]

    _plot_weight_rank_heatmaps(ds, bpp_panels, fk_panels)
